# Chapter 7 &mdash; Reversal of a DFA Yields an NFA

**Concept 11 of the Chapter 7 decomposition:** *Reversal of a DFA Yields an NFA*

Flip every arrow: final states become the initial set, the old start becomes the sole final state.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter7-NFA/Concept-Reversal-Yields-NFA/Concept-Reversal-Yields-NFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Reversing a DFA is mechanical: **flip every arrow**.

* $F$ becomes the **initial set** $Q_0$ &mdash; and since $F$ may have many states, the
  result is genuinely an **NFA**, not a DFA;
* the old $q_0$ becomes the **sole final state**;
* $\delta^R(q',a) = \{q : \delta(q,a)=q'\}$, which is a set because several states can
  share a target.

So reversal is the first operation that *forces* nondeterminism. It also proves that
regular languages are **closed under reversal** &mdash; the fact Chapter 4 used to settle
$L_{if}$.

## 2. Definitions

### A DFA and its reversal

In [ ]:
D = md2mc('''DFA
I  : 0 -> A
I  : 1 -> I
A  : 0 -> F
A  : 1 -> I
F  : 0 -> F
F  : 1 -> I
''')
R = rev_dfa(D)
print("D : q0 = %r, F = %s" % (D["q0"], sorted(D["F"])))
print("R : Q0 = %s, F = %s" % (sorted(R["Q0"]), sorted(R["F"])))

### Reversal, by hand, to see the set-valued $\delta$

In [ ]:
def reverse(D):
    Dl = {}
    for (q, a), t in D["Delta"].items():
        Dl.setdefault((t, a), set()).add(q)
    return mk_nfa(D["Q"], D["Sigma"], Dl, set(D["F"]), {D["q0"]})

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;10.&nbsp;Brzozowski's Minimization: Reverse, Determinize, Reverse, Determinize](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter7-NFA/Concept-Brzozowski-Minimization/Concept-Brzozowski-Minimization.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7-NFA/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;12.&nbsp;A Complete Illustration of Brzozowski's Minimization on `blimp`](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter7-NFA/Concept-Brzozowski-On-Blimp/Concept-Brzozowski-On-Blimp.ipynb)&nbsp;&rarr;

---

## 3. Tests

The reversed machine accepts exactly the reversed strings.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
bad = [s for s in strs if accepts_nfa(R, s) != accepts_dfa(D, s[::-1])]
print("mismatches :", len(bad))
assert not bad
print("L(R) = { w^R : w in L(D) }, verified on all %d strings" % len(strs))

The old $F$ becomes the start **set** &mdash; nondeterminism is forced.

In [ ]:
print("|F| of the original  :", len(D["F"]))
print("|Q0| of the reversal :", len(R["Q0"]))
assert R["Q0"] == D["F"]
print("\nA DFA may have many final states, so the reversal may have many start states,")
print("and a machine with several start states is by definition an NFA.")

$\delta^R$ really is set-valued where two states shared a target.

In [ ]:
mine = reverse(D)
multi = [(k, sorted(v)) for k, v in mine["Delta"].items() if len(v) > 1]
print("set-valued entries :", multi)
assert multi, "some target must have had two predecessors"

Our hand reversal agrees with `rev_dfa`.

In [ ]:
assert all(accepts_nfa(mine, s) == accepts_nfa(R, s) for s in strs)
print("hand-written reverse agrees with rev_dfa everywhere")

Closure under reversal, which Chapter 4 used on $L_{if}$.

In [ ]:
RR = nfa2dfa(rev_dfa(nfa2dfa(rev_dfa(D))))
print("reverse twice returns the original language? ", langeq_dfa(RR, D))
assert langeq_dfa(RR, D)

## 4. Animation

The reversed machine: several start states, one final state, every arrow flipped.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(R, FuseEdges=True)

## 5. Exercises


1. Reverse a DFA with exactly one final state. Is the result deterministic?
2. Prove $L^{RR} = L$ from the construction.
3. Which chapter-4 proof depended on closure under reversal?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter7-NFA/Concept-Reversal-Yields-NFA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')